# YOLOv8-Seg | Transfer Learning (COCO Pretrained)
Multi-label clothing detection + instance segmentation - 5 classes  
Backbone: CSPDarknet | Neck: PAN-FPN | Heads: Detection + Segmentation  
Init: COCO pretrained weights | Strategy: Freeze backbone, finetune neck+heads

## 1. Install & Imports

In [1]:
!pip install ultralytics --quiet

import os, json, shutil, random, glob
import numpy as np
import torch
import torch.nn as nn
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import roc_auc_score, roc_curve
from ultralytics import YOLO
import yaml

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 2. Configuration

In [ ]:
BASE_DIR      = '/kaggle/input/datasets/fashion/processed'
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train', 'images')
TRAIN_ANN_DIR = os.path.join(BASE_DIR, 'train', 'annos')
VAL_IMG_DIR   = os.path.join(BASE_DIR, 'validation', 'images')
VAL_ANN_DIR   = os.path.join(BASE_DIR, 'validation', 'annos')


YOLO_DIR = '/kaggle/working/yolo_dataset'

CAT_REMAP   = {1: 0, 2: 1, 7: 2, 8: 3, 9: 4}
CLASS_NAMES = ['short_sleeve_top', 'long_sleeve_top', 'shorts', 'trousers', 'skirt']
NUM_CLASSES = 5

EPOCHS     = 10
IMGSZ      = 640
BATCH_SIZE = 16

LR0        = 1e-3
LRF        = 1e-4
SEED       = 42
WORKERS    = 4

# Two-stage freeze strategy:
# Stage 1 (epochs 1-3): freeze backbone entirely, only train neck+heads
# Stage 2 (epochs 4-10): unfreeze all, fine-tune end-to-end with low LR
FREEZE_EPOCHS = 3
FREEZE_LAYERS = 10  # freeze first 10 layers (backbone in YOLOv8m)

CKPT_DIR = '/kaggle/working/yolo_finetune'
os.makedirs(CKPT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

n_gpu = torch.cuda.device_count()
device_str = ','.join(str(i) for i in range(n_gpu)) if n_gpu > 1 else '0'
print(f'Training device(s): {device_str}')

Training device(s): 0,1


## 3. Data Preprocessing

Identical to scratch notebook - reuses dataset if already built, otherwise rebuilds.  
Test split is carved identically (same seed) so comparisons are fair.

In [ ]:
def polygon_to_yolo_seg(polygon_pts, img_w, img_h):
    pts = np.array(polygon_pts, dtype=np.float32)
    pts[:, 0] = np.clip(pts[:, 0], 0, img_w - 1)
    pts[:, 1] = np.clip(pts[:, 1], 0, img_h - 1)
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    largest = max(contours, key=cv2.contourArea)
    if len(largest) < 3:
        return []
    coords = []
    for pt in largest.reshape(-1, 2):
        coords.append(float(np.clip(pt[0] / img_w, 0, 1)))
        coords.append(float(np.clip(pt[1] / img_h, 0, 1)))
    return coords

def parse_annotation(ann_path, img_w, img_h):
    with open(ann_path) as f:
        data = json.load(f)
    records = []
    for key, item in data.items():
        if not key.startswith('item'):
            continue
        cat_id = item.get('category_id', -1)
        if cat_id not in CAT_REMAP:
            continue
        cls = CAT_REMAP[cat_id]
        x1, y1, x2, y2 = item['bounding_box']
        x1, x2 = max(0, x1), min(img_w, x2)
        y1, y2 = max(0, y1), min(img_h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        x_c = (x1 + x2) / 2 / img_w
        y_c = (y1 + y2) / 2 / img_h
        w   = (x2 - x1) / img_w
        h   = (y2 - y1) / img_h
        seg_polys = item.get('segmentation', [])
        if not seg_polys:
            continue
        best_poly = max(seg_polys, key=lambda p: len(p))
        pts = [(best_poly[i], best_poly[i+1]) for i in range(0, len(best_poly)-1, 2)]
        if len(pts) < 3:
            continue
        seg_norm = polygon_to_yolo_seg(pts, img_w, img_h)
        if len(seg_norm) < 6:  
            continue
        records.append((cls, [x_c, y_c, w, h], seg_norm))
    return records

def write_yolo_label(label_path, records):
    with open(label_path, 'w') as f:
        for cls, bbox, seg in records:
            seg_str = ' '.join(f'{v:.6f}' for v in seg)
            f.write(f'{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} {seg_str}\n')

def build_yolo_split(img_dir, ann_dir, out_img_dir, out_lbl_dir, desc='', allowed_stems=None):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    ann_files = sorted(glob.glob(os.path.join(ann_dir, '*.json')))
    valid_files = []
    class_counts = defaultdict(int)
    skipped = 0
    for ann_path in tqdm(ann_files, desc=desc):
        stem = Path(ann_path).stem
        if allowed_stems is not None and stem not in allowed_stems:
            continue
        img_path = os.path.join(img_dir, stem + '.jpg')
        if not os.path.exists(img_path):
            skipped += 1
            continue
        img = Image.open(img_path)
        img_w, img_h = img.size
        records = parse_annotation(ann_path, img_w, img_h)
        if not records:
            skipped += 1
            continue
        out_img = os.path.join(out_img_dir, stem + '.jpg')
        if not os.path.exists(out_img):
            os.symlink(img_path, out_img)
        out_lbl = os.path.join(out_lbl_dir, stem + '.txt')
        write_yolo_label(out_lbl, records)
        for cls, _, _ in records:
            class_counts[cls] += 1
        valid_files.append(stem)
    print(f'{desc}: {len(valid_files):,} images kept, {skipped} skipped')
    return valid_files, class_counts
    
import shutil
if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)
    print('Cleared old dataset, rebuilding...')
    
if not os.path.exists(f'{YOLO_DIR}/images/train'):
    print('Building YOLO dataset from scratch...')

    all_train_valid, train_counts = build_yolo_split(
        TRAIN_IMG_DIR, TRAIN_ANN_DIR,
        f'{YOLO_DIR}/images/_train_all', f'{YOLO_DIR}/labels/_train_all', desc='Train+Test')

    random.shuffle(all_train_valid)
    n_test      = int(0.10 * len(all_train_valid))
    test_stems  = set(all_train_valid[:n_test])
    train_stems = set(all_train_valid[n_test:])

    for split_name, stems in [('train', train_stems), ('test', test_stems)]:
        os.makedirs(f'{YOLO_DIR}/images/{split_name}', exist_ok=True)
        os.makedirs(f'{YOLO_DIR}/labels/{split_name}', exist_ok=True)
        for stem in stems:
            src_img = f'{YOLO_DIR}/images/_train_all/{stem}.jpg'
            src_lbl = f'{YOLO_DIR}/labels/_train_all/{stem}.txt'
            dst_img = f'{YOLO_DIR}/images/{split_name}/{stem}.jpg'
            dst_lbl = f'{YOLO_DIR}/labels/{split_name}/{stem}.txt'
            if not os.path.exists(dst_img): os.symlink(os.path.abspath(src_img), dst_img)
            if not os.path.exists(dst_lbl): shutil.copy(src_lbl, dst_lbl)

    train_counts = defaultdict(int)
    for lbl in glob.glob(f'{YOLO_DIR}/labels/train/*.txt'):
        with open(lbl) as f:
            for line in f:
                train_counts[int(line.split()[0])] += 1

    _, _ = build_yolo_split(
        VAL_IMG_DIR, VAL_ANN_DIR,
        f'{YOLO_DIR}/images/val', f'{YOLO_DIR}/labels/val', desc='Val')

    print('Done building dataset.')
else:
    print('YOLO dataset already exists - reusing.')
    train_counts = defaultdict(int)
    for lbl in glob.glob(f'{YOLO_DIR}/labels/train/*.txt'):
        with open(lbl) as f:
            for line in f:
                train_counts[int(line.split()[0])] += 1

print(f'\nSplits present:')
for s in ['train','val','test']:
    n = len(glob.glob(f'{YOLO_DIR}/images/{s}/*.jpg'))
    print(f'  {s}: {n:,}')

Building YOLO dataset from scratch...


Train+Test: 100%|██████████| 144174/144174 [1:02:37<00:00, 38.37it/s]


Train+Test: 144,174 images kept, 0 skipped


Val: 100%|██████████| 23741/23741 [10:06<00:00, 39.13it/s]


Val: 23,741 images kept, 0 skipped
Done building dataset.

Splits present:
  train: 129,757
  val: 23,741
  test: 14,417


## 4. Class Imbalance Weights

In [4]:
counts = np.array([train_counts[i] for i in range(NUM_CLASSES)], dtype=np.float32)
pos_weights = counts.sum() / (NUM_CLASSES * counts)
pos_weights = np.clip(pos_weights, 0.5, 10.0)
mean_cls_pw = float(pos_weights.mean())

print('Per-class pos_weights:')
for name, w in zip(CLASS_NAMES, pos_weights):
    print(f'  {name:<22}: {w:.3f}')
print(f'Mean cls_pw: {mean_cls_pw:.3f}')

Per-class pos_weights:
  short_sleeve_top      : 0.643
  long_sleeve_top       : 1.282
  shorts                : 1.257
  trousers              : 0.832
  skirt                 : 1.497
Mean cls_pw: 1.102


## 5. Dataset YAML

In [5]:
yaml_path = f'{YOLO_DIR}/dataset.yaml'
if not os.path.exists(yaml_path):
    dataset_yaml = {
        'path': YOLO_DIR, 'train': 'images/train',
        'val': 'images/val', 'test': 'images/test',
        'nc': NUM_CLASSES, 'names': CLASS_NAMES
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(dataset_yaml, f, default_flow_style=False)
print(open(yaml_path).read())

names:
- short_sleeve_top
- long_sleeve_top
- shorts
- trousers
- skirt
nc: 5
path: /kaggle/working/yolo_dataset
test: images/test
train: images/train
val: images/val



## 6. Model - YOLOv8-seg with COCO Pretrained Weights

**Transfer learning strategy:**
- Load `yolov8m-seg.pt` - pretrained on COCO 80-class detection+segmentation
- Replace detection/seg heads for 5 classes (Ultralytics does this automatically)
- Stage 1: freeze backbone (layers 0-9), train neck+heads only for 3 epochs
- Stage 2: unfreeze all, end-to-end fine-tune with lower LR for remaining epochs

**Why this is better than freezing for all 10 epochs:**  
Stage 1 adapts the heads to our 5-class domain without destroying pretrained features.  
Stage 2 lets the backbone also adapt to clothing-specific textures.

In [ ]:
# Stage 1: Train with backbone frozen
print(f'Stage 1: Freeze backbone, train neck+heads for {FREEZE_EPOCHS} epochs')
print(f'Downloading yolov8m-seg.pt (COCO pretrained)...')

model = YOLO('yolov8m-seg.pt')

total_params = sum(p.numel() for p in model.model.parameters())
print(f'Total parameters: {total_params/1e6:.1f}M')
os.environ["TQDM_DISABLE"] = "1"
os.environ["ULTRALYTICS_QUIET"] = "1"
stage1_results = model.train(
    data      = yaml_path,
    epochs    = FREEZE_EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH_SIZE,
    device    = device_str,
    workers   = WORKERS,
    project   = CKPT_DIR,
    name      = 'finetune_stage1',
    exist_ok  = True,

    optimizer = 'AdamW',      # AdamW preferred for transfer learning
    lr0       = LR0,
    lrf       = 0.1,
    weight_decay = 1e-4,
    warmup_epochs = 1.0,      # less warmup - pretrained weights already stable

    freeze    = FREEZE_LAYERS,  # freeze first N layers (backbone)

    cls       = 0.5,
    box       = 7.5,
    dfl       = 1.5,

    # Lighter augmentation in stage 1 - backbone is frozen, no point in heavy aug
    mosaic    = 0.5,
    mixup     = 0.0,
    degrees   = 0.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    flipud    = 0.0,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,

    val       = True,
    save      = True,
    save_period = 1,
    plots     = True,
    seed      = SEED,
)

print('Stage 1 complete.')

Stage 1: Freeze backbone, train neck+heads for 3 epochs
Total parameters: 27.3M
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8

In [ ]:
# Stage 2: Load stage1 best, unfreeze all, end-to-end fine-tune
remaining_epochs = EPOCHS - FREEZE_EPOCHS
print(f'Stage 2: Unfreeze all layers, end-to-end for {remaining_epochs} epochs')

stage1_best = f'{CKPT_DIR}/finetune_stage1/weights/best.pt'
model2 = YOLO(stage1_best)
os.environ["TQDM_DISABLE"] = "1"
os.environ["ULTRALYTICS_QUIET"] = "1"
stage2_results = model2.train(
    data      = yaml_path,
    epochs    = remaining_epochs,
    imgsz     = IMGSZ,
    batch     = BATCH_SIZE,
    device    = device_str,
    workers   = WORKERS,
    project   = CKPT_DIR,
    name      = 'finetune_stage2',
    exist_ok  = True,

    optimizer = 'AdamW',
    lr0       = LR0 * 0.3,   # lower LR for end-to-end fine-tune
    lrf       = 0.1,
    weight_decay = 1e-4,
    warmup_epochs = 0.5,

    freeze    = 0,           # no freezing - all layers trainable

    cls       = 0.5,
    box       = 7.5,
    dfl       = 1.5,        
    copy_paste = 0.3,         
    erasing    = 0.4,

    # Full augmentation in stage 2
    mosaic    = 1.0,
    mixup     = 0.1,
    degrees   = 0.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    flipud    = 0.0,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,

    val       = True,
    save      = True,
    save_period = 1,
    plots     = True,
    seed      = SEED,
)

best_ckpt = f'{CKPT_DIR}/finetune_stage2/weights/best.pt'
print(f'\nBest checkpoint: {best_ckpt}')
print(f'Best mAP50-95: {stage2_results.results_dict.get("metrics/mAP50-95(B)", "N/A")}')

Stage 2: Unfreeze all layers, end-to-end for 7 epochs
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=7, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/kaggle/working/yolo_finetune/fi

AttributeError: 'NoneType' object has no attribute 'results_dict'

## 7. Final Evaluation on Test Set

In [ ]:
best_ckpt = '/kaggle/input/models/ankithkini/best-output/pytorch/default/1/best.pt'
best_model = YOLO(best_ckpt)

test_results = best_model.val(
    data     = yaml_path,
    split    = 'test',
    imgsz    = IMGSZ,
    batch    = BATCH_SIZE,
    device   = device_str,
    verbose  = False,
    plots    = True,
    save_json = True,
    project  = CKPT_DIR,
    name     = 'test_eval',
    exist_ok = True,
)

print('\n' + '='*70)
print('TEST SET RESULTS - YOLOv8-seg (Transfer Learning)')
print('='*70)
rd = test_results.results_dict
print(f"{'Metric':<35} {'Value':>10}")
print('-'*46)
for k, v in rd.items():
    if isinstance(v, float):
        print(f'{k:<35} {v:>10.4f}')

## 8. Per-Class Detection Metrics

In [ ]:
def print_per_class_detection(results_obj, split_name='Test'):
    maps   = results_obj.box.maps
    maps50 = results_obj.box.ap50
    prec   = results_obj.box.p
    rec    = results_obj.box.r

    print(f'\n{split_name} - Per-Class Detection (Bounding Box)')
    print(f'{"Class":<22} {"Precision":>10} {"Recall":>10} {"mAP50":>10} {"mAP50-95":>10} {"F1":>10}')
    print('-'*74)
    for i, name in enumerate(CLASS_NAMES):
        p  = prec[i]  if i < len(prec)  else 0
        r  = rec[i]   if i < len(rec)   else 0
        m50  = maps50[i] if i < len(maps50) else 0
        m    = maps[i]   if i < len(maps)   else 0
        f1 = 2*p*r/(p+r+1e-8)
        print(f'{name:<22} {p:>10.4f} {r:>10.4f} {m50:>10.4f} {m:>10.4f} {f1:>10.4f}')
    print('-'*74)
    macro_f1 = np.mean([2*prec[i]*rec[i]/(prec[i]+rec[i]+1e-8)
                        for i in range(min(len(prec), NUM_CLASSES))])
    print(f'{"Macro-avg":<22} {np.mean(prec):>10.4f} {np.mean(rec):>10.4f} '
          f'{np.mean(maps50):>10.4f} {np.mean(maps):>10.4f} {macro_f1:>10.4f}')

    print(f'\n{split_name} - Per-Class Segmentation (Mask)')
    if hasattr(results_obj, 'seg') and results_obj.seg is not None:
        maps_seg  = results_obj.seg.maps
        maps50_seg = results_obj.seg.ap50
        prec_seg  = results_obj.seg.p
        rec_seg   = results_obj.seg.r
        print(f'{"Class":<22} {"Precision":>10} {"Recall":>10} {"mAP50":>10} {"mAP50-95":>10}')
        print('-'*64)
        for i, name in enumerate(CLASS_NAMES):
            p  = prec_seg[i]  if i < len(prec_seg) else 0
            r  = rec_seg[i]   if i < len(rec_seg)  else 0
            m50 = maps50_seg[i] if i < len(maps50_seg) else 0
            m   = maps_seg[i]   if i < len(maps_seg)   else 0
            print(f'{name:<22} {p:>10.4f} {r:>10.4f} {m50:>10.4f} {m:>10.4f}')
        print('-'*64)
        print(f'{"Macro-avg":<22} {np.mean(prec_seg):>10.4f} {np.mean(rec_seg):>10.4f} '
              f'{np.mean(maps50_seg):>10.4f} {np.mean(maps_seg):>10.4f}')

print_per_class_detection(test_results, split_name='TEST')

## 9. mIoU and Dice on Test Set

In [ ]:
def polygon_to_mask(polygon_flat, img_h, img_w):
    pts = np.array(polygon_flat).reshape(-1, 2)
    pts[:, 0] *= img_w
    pts[:, 1] *= img_h
    pts = pts.astype(np.int32)
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    return mask

def compute_iou_dice(pred_mask, gt_mask):
    inter = (pred_mask & gt_mask).sum()
    union = (pred_mask | gt_mask).sum()
    iou  = inter / (union + 1e-8)
    dice = 2 * inter / (pred_mask.sum() + gt_mask.sum() + 1e-8)
    return float(iou), float(dice)

def evaluate_miou_dice(model, test_img_dir, test_lbl_dir, imgsz=640, conf=0.25, iou_thresh=0.5):
    per_class_iou  = defaultdict(list)
    per_class_dice = defaultdict(list)
    test_images = sorted(glob.glob(os.path.join(test_img_dir, '*.jpg')))
    print(f'Running mIoU/Dice on {len(test_images)} test images...')
    for img_path in tqdm(test_images):
        stem = Path(img_path).stem
        lbl_path = os.path.join(test_lbl_dir, stem + '.txt')
        if not os.path.exists(lbl_path):
            continue
        img = cv2.imread(img_path)
        img_h, img_w = img.shape[:2]
        gt_instances = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                cls = int(parts[0])
                seg_coords = list(map(float, parts[5:]))
                if len(seg_coords) >= 6:
                    gt_mask = polygon_to_mask(seg_coords, img_h, img_w)
                    gt_instances.append((cls, gt_mask))
        if not gt_instances:
            continue
        result = model(img_path, imgsz=imgsz, conf=conf, verbose=False)[0]
        if result.masks is None:
            for cls, _ in gt_instances:
                per_class_iou[cls].append(0.0)
                per_class_dice[cls].append(0.0)
            continue
        pred_masks = result.masks.data.cpu().numpy()
        pred_cls   = result.boxes.cls.cpu().numpy().astype(int)
        resized_masks = [cv2.resize(m.astype(np.uint8), (img_w, img_h),
                         interpolation=cv2.INTER_NEAREST) for m in pred_masks]
        matched_preds = set()
        for gt_cls, gt_mask in gt_instances:
            best_iou, best_idx = 0.0, -1
            for pidx, (pcls, pmask) in enumerate(zip(pred_cls, resized_masks)):
                if pidx in matched_preds or pcls != gt_cls:
                    continue
                iou, _ = compute_iou_dice(pmask, gt_mask)
                if iou > best_iou:
                    best_iou, best_idx = iou, pidx
            if best_idx >= 0 and best_iou >= iou_thresh:
                _, dice = compute_iou_dice(resized_masks[best_idx], gt_mask)
                per_class_iou[gt_cls].append(best_iou)
                per_class_dice[gt_cls].append(dice)
                matched_preds.add(best_idx)
            else:
                per_class_iou[gt_cls].append(0.0)
                per_class_dice[gt_cls].append(0.0)
    return per_class_iou, per_class_dice

per_class_iou, per_class_dice = evaluate_miou_dice(
    best_model, f'{YOLO_DIR}/images/test', f'{YOLO_DIR}/labels/test',conf=0.1)

print('\n' + '='*56)
print('Segmentation: mIoU and Dice - per class')
print('='*56)
print(f'{"Class":<22} {"mIoU":>10} {"Dice":>10} {"N":>8}')
print('-'*52)
for cls_id, name in enumerate(CLASS_NAMES):
    ious  = per_class_iou[cls_id]
    dices = per_class_dice[cls_id]
    print(f'{name:<22} {np.mean(ious) if ious else 0:>10.4f} '
          f'{np.mean(dices) if dices else 0:>10.4f} {len(ious):>8}')
print('-'*52)
macro_iou  = np.mean([np.mean(per_class_iou[i])  for i in range(NUM_CLASSES) if per_class_iou[i]])
macro_dice = np.mean([np.mean(per_class_dice[i]) for i in range(NUM_CLASSES) if per_class_dice[i]])
print(f'{"Macro-avg":<22} {macro_iou:>10.4f} {macro_dice:>10.4f}')

In [ ]:
# Task 3.2 - Scratch vs Transfer Learning Comparison
print('\n' + '='*70)
print('TASK 3.2 - Scratch vs Transfer Learning Comparison')
print('='*70)

scratch_metrics = {
    'mAP50 (box)':    0.000,   
    'mAP50-95 (box)': 0.000,   
    'Macro Dice':     0.000,   
}

rd = test_results.results_dict
transfer_metrics = {
    'mAP50 (box)':    rd.get('metrics/mAP50(B)', 0),
    'mAP50-95 (box)': rd.get('metrics/mAP50-95(B)', 0),
    'Macro Dice':     macro_dice,
}

print(f"\n{'Metric':<22} {'Scratch':>12} {'Transfer':>12} {'Delta':>10}")
print('-'*58)
for k in scratch_metrics:
    s = scratch_metrics[k]
    t = transfer_metrics[k]
    print(f'{k:<22} {s:>12.4f} {t:>12.4f} {t-s:>+10.4f}')

## 10. ROC Curves and AUC

In [ ]:
def compute_roc_auc_detection(model, test_img_dir, test_lbl_dir, imgsz=640):
    all_scores = defaultdict(list)
    all_labels = defaultdict(list)
    test_images = sorted(glob.glob(os.path.join(test_img_dir, '*.jpg')))
    print(f'Computing ROC/AUC on {len(test_images)} test images...')
    for img_path in tqdm(test_images):
        stem = Path(img_path).stem
        lbl_path = os.path.join(test_lbl_dir, stem + '.txt')
        if not os.path.exists(lbl_path):
            continue
        gt_classes = set()
        with open(lbl_path) as f:
            for line in f:
                gt_classes.add(int(line.strip().split()[0]))
        result = model(img_path, imgsz=imgsz, conf=0.001, verbose=False)[0]
        pred_conf_per_class = defaultdict(float)
        if result.boxes is not None and len(result.boxes):
            for conf, cls in zip(result.boxes.conf.cpu().numpy(),
                                 result.boxes.cls.cpu().numpy().astype(int)):
                pred_conf_per_class[cls] = max(pred_conf_per_class[cls], float(conf))
        for cls_id in range(NUM_CLASSES):
            all_scores[cls_id].append(pred_conf_per_class[cls_id])
            all_labels[cls_id].append(1 if cls_id in gt_classes else 0)
    return all_scores, all_labels

roc_scores, roc_labels = compute_roc_auc_detection(
    best_model, f'{YOLO_DIR}/images/test', f'{YOLO_DIR}/labels/test')

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(22, 4))
print(f'\n{"Class":<22} {"AUC":>8}')
print('-'*32)
aucs = []
for i, (name, ax) in enumerate(zip(CLASS_NAMES, axes)):
    y_true  = np.array(roc_labels[i])
    y_score = np.array(roc_scores[i])
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        continue
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    aucs.append(auc)
    ax.plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC={auc:.3f}')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=9)
    print(f'{name:<22} {auc:>8.4f}')
if aucs:
    print(f'{"Macro-avg":<22} {np.mean(aucs):>8.4f}')
plt.suptitle('ROC Curves - YOLOv8-seg Transfer Learning (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/roc_finetune_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Training Curves (Both Stages)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_pairs = [
    ('train/box_loss', 'val/box_loss', 'Box Loss'),
    ('train/cls_loss', 'val/cls_loss', 'Cls Loss'),
    ('train/seg_loss', 'val/seg_loss', 'Seg Loss'),
    ('metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'Det mAP'),
    ('metrics/mAP50(M)', 'metrics/mAP50-95(M)', 'Seg mAP'),
    ('lr/pg0', None, 'Learning Rate'),
]

dfs = []
for stage, csv_path in [
    ('finetune_stage1', '/kaggle/input/datasets/ankithkini/results1/results.csv'),
    ('finetune_stage2', '/kaggle/input/datasets/ankithkini/results2/results.csv'),
]:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        dfs.append(df)

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    for ax, (col1, col2, title) in zip(axes.flat, plot_pairs):
        if col1 in df_all.columns:
            ax.plot(df_all[col1], label=col1.split('/')[0])
            ax.axvline(x=FREEZE_EPOCHS-1, color='red', linestyle=':', alpha=0.5, label='Unfreeze')
        if col2 and col2 in df_all.columns:
            ax.plot(df_all[col2], label=col2.split('/')[0], linestyle='--')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
    plt.suptitle('Training Curves - YOLOv8-seg Transfer Learning', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{CKPT_DIR}/curves_finetune.png', dpi=150, bbox_inches='tight')
    plt.show()

## 12. Qualitative Visualisation

In [ ]:
COLORS = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
test_images = sorted(glob.glob(f'{YOLO_DIR}/images/test/*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flat, test_images):
    result = best_model(img_path, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    overlay = img_rgb.copy().astype(np.float32)
    if result.masks is not None:
        h, w = img_rgb.shape[:2]
        for mask, cls_id in zip(result.masks.data.cpu().numpy(),
                                result.boxes.cls.cpu().numpy().astype(int)):
            m = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            color = np.array(COLORS[cls_id % NUM_CLASSES][:3]) * 255
            overlay[m == 1] = overlay[m == 1] * 0.5 + color * 0.5
    ax.imshow(overlay.astype(np.uint8))
    if result.boxes is not None and len(result.boxes):
        for box, cls_id, conf in zip(result.boxes.xyxy.cpu().numpy(),
                                      result.boxes.cls.cpu().numpy().astype(int),
                                      result.boxes.conf.cpu().numpy()):
            x1,y1,x2,y2 = box.astype(int)
            color = COLORS[cls_id % NUM_CLASSES][:3]
            rect = mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                                       linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-4, f'{CLASS_NAMES[cls_id]} {conf:.2f}',
                    color=color, fontsize=7, fontweight='bold')
    ax.axis('off')
    ax.set_title(Path(img_path).stem, fontsize=8)
plt.suptitle('Qualitative Results - YOLOv8-seg Transfer Learning', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/qualitative_finetune.png', dpi=150, bbox_inches='tight')
plt.show()